# Specialized Quantities

Beyond `astropy.units.Quantity`, astropy provides further specialized quantities. `astropy-xarray` captures this using the `units.class` attribute.

In [ ]:
import xarray as xr
from astropy import units as u

## Time and TimeDelta

`astropy_xarray` patches `astropy.time.Time` and `astropy.time.TimeDelta` to be recognized xarray, capturing:
* format
* timescale
* precision

In [ ]:
from astropy.time import Time, TimeDelta

import astropy_xarray  # noqa: F401

ds = xr.Dataset(
    data_vars={
        "d_time": (
            "time",
            Time([0, 1, 2], format="unix", scale="utc").copy("datetime64"),
        ),
        "s_time": (
            "time",
            Time([0, 1, 2], format="unix", scale="utc").copy("isot"),
        ),
        "interval": ("time", TimeDelta([0.9, 0.9, 0.9], format="sec", scale="tai")),
    },
    coords=xr.Coordinates(
        {
            "f_time": ("time", Time([0, 1, 2], format="unix", scale="utc")),
        }
    ),
)
ds

In [ ]:
# dequantified attributes
ds.astropy.dequantify()

## SkyCoord, Frame and Angle

SkyCoord and Frame fully support conversion to datasets preserving:
* Representation
* Differential
* Frame
* Angle, Longitude, Latitude containers

Conversion optionally requires coords argument for applying named coordinates to data variable axes.

Frame-specific mapped names for data variables coming soon.

In [ ]:
from astropy.coordinates import SkyCoord

from astropy_xarray.coordinates.sky_coord import (
    skycoord_to_dataset,
)

sc = SkyCoord(
    ra=[2, 6, 7, 4] * u.deg,
    dec=[4, 7, 4, 3] * u.deg,
)


s = skycoord_to_dataset(sc, coords={"field": ["a", "b", "c", "d"]})
display(s)

## Logarithmic Units and Quantities

Astropy provides a generic `LogQuantity` and other various log scale specialized classes and units:

$$scalar = A * log_{B}(\frac{V}{V_0})$$


| Name                | Equiv SI Unit | Scale Factor   | Log Base  | 0 Reference                            |
| --------------------| ------------- | -------------- | --------- | ---------------------------------------|
| Magnitude (absolute)| Unitless      | -2.5           | 10        | Apparent brightness at 10pc away       |
| Dex                 | Unitless      | 1              | 10        | 1                                      |
| Decibels (power)    | Unitless      | 10             | 10        | Variable                               |
| Decibels (amplitude)| Unitless      | 20             | 10        | Variable                               |
| Magnitude (apparent)| Unitless      | -2.5           | 10        | Usually Vega or AB from Earth          |
| Magnitude (AB)      | Jy            | -2.5           | 10        | 3631 Jy                                |

In [ ]:
import types

import astropy.units
import astropy.units as u
import numpy as np
from astropy.units import Dex, Equivalency, Magnitude, Quantity

import astropy_xarray  # noqa: F401


# extend registry
def asplund():
    asplund = types.ModuleType("asplund")
    astropy.units.def_unit(
        ["solH", "Xsun"],
        format={"latex": "X_{\\odot}", "unicode": "X⊙"},
        namespace=asplund.__dict__,
    )
    astropy.units.def_unit(
        ["solHe", "Ysun"],
        format={"latex": "Y_{\\odot}", "unicode": "Y⊙"},
        namespace=asplund.__dict__,
    )
    astropy.units.def_unit(
        ["solMetal", "Zsun"],
        format={"latex": "Z_{\\odot}", "unicode": "Z⊙"},
        namespace=asplund.__dict__,
    )

    return asplund


if not hasattr(u, "asplund"):
    u.asplund = asplund()
    u.add_enabled_units(u.asplund)
    u.__dict__.update(u.get_current_unit_registry().registry)


def asplund_solar_relative_mass() -> Equivalency:
    """measurements from https://www.aanda.org/articles/aa/full_html/2021/09/aa40445-21/aa40445-21.html"""
    return Equivalency(
        [
            (u.Xsun, u.Msun, lambda x: x * 0.7438),
            (u.Ysun, u.Msun, lambda x: x * 0.2423),
            (u.Zsun, u.Msun, lambda x: x * 0.0139),
        ]
    )


if not hasattr(u, "asplund_solar_relative_mass"):
    u.asplund_solar_relative_mass = asplund_solar_relative_mass
    # uncomment for implicit conversions
    # u.add_enabled_equivalencies(u.asplund_solar_relative_mass())

In [ ]:
ds = xr.Dataset(
    coords=xr.Coordinates(
        {
            "timestamp": ("time", Time([1.7e9], format="unix")),
            "body_label": ("body", ["sun", "proxima centauri"]),
            "wavelength": ("frequency", [565, 535, 445] * u.nm),
        }
    ),
    data_vars={
        "mass": (["time", "body"], Dex([[0.0, 0.02]], unit=u.dex(u.Msun))),
        "distance": (["time", "body"], [[1.007, 12950]] * u.AU),
        "luminosity": (
            ["time", "body"],
            Quantity([[1, 0.001]], unit=u.Lsun).to(u.dex(u.Lsun)),
        ),  # radiant flux
        "metallicity": (["time", "body"], Dex([[0.0, np.nan]], unit=u.dex(u.Zsun))),
        "apparent_magnitude": (
            ["time", "body"],
            Magnitude([[-26.74, 10.67]]),
        ),  # to Vega
        "absolute_magnitude": (["time", "body"], Magnitude([[4.83, 15.60]])),
        # "irradiance": (["time", "body"], [[1361, 0]] * u.W / u.m**2),
        # "radiant_flux_density": (["time", "body"], [[1.0, 0.0]] * u.W / u.m**2),
        # "spectral_flux_density": (["time", "body", "frequency"], [[[68000, 0, 0], [0, 0, 0]]]  * u.Jy),
    },
)
ds

In [ ]:
ds.astropy.sel(body=[0]).astropy.to(
    {
        "mass": u.kg,
        "distance": u.km,
        "illuminance": u.cd * u.sr / u.m**2,
        "luminosity": u.kW,
        "metallicity": u.kg,
    },
    equivalencies=asplund_solar_relative_mass(),
)